# Day 4 — HOL 2: Partitioning, Z-ORDER / Liquid Clustering & Caching
### GlobalMart Data Engineering · 3:30 PM – 5:30 PM

---

## What We Are Building

A small, disposable practice table modeled on `gbmart.bronze.customers`, deliberately written as many tiny files — then we apply each performance lever from ILT 3 and *measure* the before/after difference ourselves, instead of just reading about it.

> **Why Bronze, not Gold?** By Day 4, only the Bronze layer exists — Silver (Day 5), the dimension tables (Day 6), and `fact_sales` (Day 7) haven't been built yet. Everything in this HOL reads from `gbmart.bronze.*` for that reason; you'll get to repeat these exact same performance levers against Gold tables once they exist.

## By the End of This HOL You Will Have

- Created a table with the small-file problem on purpose, and measured its effect on file count and scan time
- Run `OPTIMIZE` and `OPTIMIZE ... ZORDER BY`, and measured the improvement
- Created a Liquid Clustering table as an alternative to Z-ORDER, and compared the two
- Cached a small lookup table and measured repeated-access speedup

---

## Before You Start — Cost & Safety Note

> **Everything in this HOL runs against a personal practice table in your own catalog/schema — never against the shared `gbmart.bronze.*` / `gbmart.silver.*` / `gbmart.gold.*` tables.** `OPTIMIZE`, `ZORDER`, and especially `VACUUM` rewrite or delete real files; running them against shared cohort tables would affect everyone else's work. Replace `YOUR_SCHEMA` below with a schema that is actually yours (for example `main.<your_name>`) before running anything. Do not run any `VACUUM` command with `RETAIN 0 HOURS` — it permanently deletes file history and cannot be undone. Nothing in this HOL starts a cluster, job, or pipeline — it all runs as ordinary cells on your already-running cluster.

In [ ]:
# ─── SETUP ─────────────────────────────────────────────────────────────────────────
# Replace YOUR_SCHEMA with a schema you own — never point this at gbmart.*
import time
from pyspark.sql.functions import col, current_timestamp, rand

PRACTICE_SCHEMA = "main.YOUR_SCHEMA"   # ← e.g. "main.virinchy_practice"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PRACTICE_SCHEMA}")

PRACTICE_TABLE = f"{PRACTICE_SCHEMA}.orders_partitioning_practice"
print(f"Practice table target: {PRACTICE_TABLE}")

---
## Phase 1 — Create the Small-File Problem on Purpose

We read the real `gbmart.bronze.customers` table (read-only — safe) and deliberately re-partition it into many tiny Spark partitions before writing, so our practice table starts life with hundreds of small files, exactly like an unmanaged streaming table would.

In [ ]:
# ─── Read real bronze.customers (read-only) and inflate it into a small-file table ──
# repartition(200) forces 200 output files even though the data doesn't need it —
# this simulates months of small streaming writes in one step, for teaching purposes.

source_df = spark.table("gbmart.bronze.customers")
print(f"Source rows: {source_df.count():,}")

(
    source_df
    .repartition(200)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PRACTICE_TABLE)
)

file_count = spark.sql(f"DESCRIBE DETAIL {PRACTICE_TABLE}").select("numFiles").collect()[0][0]
print(f"Practice table written with {file_count} files (deliberately fragmented).")

---
## Phase 2 — Measure the Baseline

Time a filtered query against the fragmented table, and note the file count. We'll compare both numbers after each optimization.

In [ ]:
# ─── Baseline timing ───────────────────────────────────────────────────────────
# Pick any CustomerID present in your data — replace CUSTOMER_ID_HERE.

CUSTOMER_ID_HERE = spark.table(PRACTICE_TABLE).select("CustomerID").first()[0]
print(f"Filtering on CustomerID = {CUSTOMER_ID_HERE}")

start = time.time()
result = spark.table(PRACTICE_TABLE).filter(col("CustomerID") == CUSTOMER_ID_HERE).count()
baseline_seconds = time.time() - start

print(f"Matching rows : {result}")
print(f"Baseline query time : {baseline_seconds:.3f}s")
print(f"File count before OPTIMIZE : {file_count}")

---
## Phase 3 — OPTIMIZE + Z-ORDER

Compact the small files into large ones, and Z-ORDER by the column we're filtering on.

In [ ]:
# ─── Run OPTIMIZE with ZORDER BY the filter column ────────────────────────
spark.sql(f"OPTIMIZE {PRACTICE_TABLE} ZORDER BY (CustomerID)")

file_count_after = spark.sql(f"DESCRIBE DETAIL {PRACTICE_TABLE}").select("numFiles").collect()[0][0]
print(f"File count after OPTIMIZE + ZORDER : {file_count_after}  (was {file_count})")

In [ ]:
# ─── Re-measure the same filtered query ────────────────────────────────────
start = time.time()
result = spark.table(PRACTICE_TABLE).filter(col("CustomerID") == CUSTOMER_ID_HERE).count()
zorder_seconds = time.time() - start

print(f"Matching rows : {result}")
print(f"Query time after OPTIMIZE+ZORDER : {zorder_seconds:.3f}s  (baseline was {baseline_seconds:.3f}s)")
print(f"On a small practice table the gap may be modest — the effect grows dramatically at real Bronze/Gold scale (millions of rows).")

---
## Phase 4 — Liquid Clustering as an Alternative

Instead of Z-ORDER on an existing table, create a **new** table with `CLUSTER BY` — Liquid Clustering's incremental approach means future `OPTIMIZE` calls stay cheap even as new data arrives, without re-specifying ZORDER columns each time.

In [ ]:
# ─── Create a Liquid Clustering version of the same practice data ──────────────
LIQUID_TABLE = f"{PRACTICE_SCHEMA}.orders_liquid_practice"

spark.sql(f"DROP TABLE IF EXISTS {LIQUID_TABLE}")
spark.sql(f"""
    CREATE TABLE {LIQUID_TABLE}
    CLUSTER BY (CustomerID)
    AS SELECT * FROM {PRACTICE_TABLE}
""")

# OPTIMIZE on a clustered table applies clustering incrementally — no ZORDER BY needed
spark.sql(f"OPTIMIZE {LIQUID_TABLE}")

start = time.time()
result = spark.table(LIQUID_TABLE).filter(col("CustomerID") == CUSTOMER_ID_HERE).count()
liquid_seconds = time.time() - start

print(f"Matching rows : {result}")
print(f"Liquid Clustering query time : {liquid_seconds:.3f}s")
print()
print(f"{'Baseline (fragmented)':<28} {baseline_seconds:.3f}s")
print(f"{'OPTIMIZE + ZORDER':<28} {zorder_seconds:.3f}s")
print(f"{'Liquid Clustering':<28} {liquid_seconds:.3f}s")

---
## Phase 5 — Caching

A small dimension-shaped table read repeatedly is the textbook caching case. We simulate "repeatedly" by running the same aggregation multiple times, with and without `.cache()`.

In [ ]:
# ─── Without cache: every access re-reads from storage ──────────────────────
# gbmart.bronze.payment_methods is small and lookup-shaped (a handful of rows) —
# exactly the kind of table that gets read repeatedly in a real pipeline.
dim_df = spark.table("gbmart.bronze.payment_methods")

start = time.time()
for _ in range(5):
    dim_df.count()
uncached_seconds = time.time() - start
print(f"5x count() WITHOUT cache: {uncached_seconds:.3f}s total")

In [ ]:
# ─── With cache: first access materializes it, rest read from memory ────────────
dim_df_cached = spark.table("gbmart.bronze.payment_methods").cache()
dim_df_cached.count()   # first access — triggers caching, not part of the timed comparison

start = time.time()
for _ in range(5):
    dim_df_cached.count()
cached_seconds = time.time() - start
print(f"5x count() WITH cache: {cached_seconds:.3f}s total  (was {uncached_seconds:.3f}s uncached)")

dim_df_cached.unpersist()   # always free cache memory when you're done with it
print("Cache released.")

---
## Cleanup (Optional)

Run this once you're done experimenting — it removes the two disposable practice tables so they don't linger in your schema.

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {PRACTICE_TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {LIQUID_TABLE}")
# print("Practice tables dropped.")

---
## Key Takeaways

1. **The small-file problem is easy to create by accident and easy to fix on purpose** — `repartition()` before a write is a common accidental cause; `OPTIMIZE` is the fix
2. **Z-ORDER and Liquid Clustering both improve filtered-query performance** by co-locating related rows — Liquid Clustering is the modern default for new tables because it clusters incrementally
3. **Caching helps when the same DataFrame is read repeatedly** in one session — it does nothing for a single scan, and costs cluster memory, so always `.unpersist()` when done
4. **All of this ran against a personal practice table** — the same commands against `gbmart.bronze.*` / `gbmart.gold.*` would affect the whole cohort's shared data, which is why production `OPTIMIZE`/`VACUUM` runs are scheduled jobs (Databricks Workflows, covered later), not ad-hoc notebook cells

---

## Discussion Questions

1. *Why did we use `repartition(200)` before writing the practice table? What real-world scenario does this simulate?*
2. *If the practice table only has a few thousand rows, why might the ZORDER speedup look small compared to what ILT 3 described for GlobalMart's real Bronze tables?*
3. *What is the risk of running `OPTIMIZE` or `VACUUM` directly against `gbmart.bronze.orders` during a live class session, versus running it against your own `main.YOUR_SCHEMA` copy?*
4. *Why do we call `.unpersist()` after the caching experiment instead of leaving the DataFrame cached?*
5. *Liquid Clustering doesn't require specifying ZORDER columns on every `OPTIMIZE` call. Why does that matter for a table whose query patterns change over time?*